# FLUX Image Generation

Generates blog header/card images via the Azure FLUX endpoint.

All secrets and config come from a local `.env` file (see `.env.example`). Generated images
are written to `IMAGE_OUTPUT_DIR` (default `../generated-images`, **outside** this folder and
gitignored). 

In [ ]:
import os
import base64
from io import BytesIO
from pathlib import Path

import requests
from dotenv import load_dotenv
from PIL import Image

load_dotenv()

# Config / secrets from .env (never hardcode keys here)
FLUX_API_KEY = os.getenv("AZURE_FLUX_API_KEY") or os.getenv("fluxkey")
GENERATION_URL = os.getenv(
    "AZURE_FLUX_GENERATION_URL",
    "https://gipiti-resource.services.ai.azure.com/providers/blackforestlabs/v1/flux-2-pro?api-version=preview",
)
OUTPUT_DIR = Path(os.getenv("IMAGE_OUTPUT_DIR", "../generated-images"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not FLUX_API_KEY:
    raise RuntimeError("AZURE_FLUX_API_KEY is not set. Copy .env.example to .env and fill it in.")

In [ ]:
prompt = """calm animestyle picture of windturbines in context of a old industrial factory which is located far in the background"""

In [ ]:
def decode_and_save_image(b64_data, output_path):
    image = Image.open(BytesIO(base64.b64decode(b64_data)))
    image.save(output_path)
    return image


def generate_image(prompt, filename_prefix, width=1408, height=800):
    body = {
        "prompt": prompt,
        "n": 1,
        "width": width,
        "height": height,
        "output_format": "png",
        "model": "FLUX.2-pro",
    }
    response = requests.post(
        GENERATION_URL,
        headers={"api-key": FLUX_API_KEY, "Content-Type": "application/json"},
        json=body,
    )
    response.raise_for_status()
    data = response.json()
    b64_img = data["data"][0]["b64_json"]
    output_path = OUTPUT_DIR / f"{filename_prefix}.png"
    decode_and_save_image(b64_img, output_path)
    print(f"Image saved to: {output_path}")
    return output_path

In [ ]:
generate_image(prompt, "generated_image")